In [147]:
import pandas as pd
from pathlib import Path

In [148]:
data_dir = Path.cwd().parent / "data"

In [149]:
arquivos = [f"despesa_ceaps_{ano}.csv" for ano in range(2013, 2023)]

despesa_ceaps = pd.concat([pd.read_csv(data_dir / arquivo, sep=";", encoding="latin1", skiprows=1)
                            for arquivo in arquivos],
                            ignore_index=True)

despesa_ceaps

,ANO,MES,SENADOR,TIPO_DESPESA,CNPJ_CPF,FORNECEDOR,DOCUMENTO,DATA,DETALHAMENTO,VALOR_REEMBOLSADO,COD_DOCUMENTO
0,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",05.914.650/0001-66,CENTRAIS ELÉTRICAS DE RONDÔNIA S.A. - CERON,000077305,18/01/2013,NaN,"214,79",705442.0
1,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",05.914.650/0001-66,CENTRAIS ELÉTRICAS DE RONDÔNIA S.A. - CERON,000077306,21/01/2013,NaN,"57,34",705443.0
2,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",004.948.028-63,GILBERTO PISELO DO NASCIMENTO,0003,30/01/2013,NaN,5000,682654.0
3,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",76.535.764/0001-43,OI S.A.,1301.000006499,14/01/2013,NaN,"398,1",705441.0
4,2013,1,ACIR GURGACZ,"Locomoção, hospedagem, alimentação, combustíve...",84.707.538/0001-20,YURI COMÉRCIO DE COMBUSTÍVEIS LTDA,000.004.274,23/01/2013,NaN,1128,682650.0
...,...,...,...,...,...,...,...,...,...,...,...
225068,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",22.052.777/0001-32,Exceller Tour,WIXHAI,06/12/2022,"Companhia Aérea: LATAM, Localizador: WIXHAI. P...","2893,04",2191398.0
225069,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",22.052.777/0001-32,Exceller Tour,WITOLM,09/12/2022,"Companhia Aérea: GOL, Localizador: WITOLM. Pas...","1180,19",2192272.0
225070,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",22.052.777/0001-32,Exceller Tour,THPKVQ,20/12/2022,"Companhia Aérea: TAM, Localizador: THPKVQ. Pas...","2671,9",2192274.0
225071,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",22.052.777/0001-32,Exceller Tour,QNN9HX,21/12/2022,"Companhia Aérea: AZUL, Localizador: QNN9HX. Pa...","1334,31",2192244.0


In [150]:
despesa_ceaps = despesa_ceaps.drop(['CNPJ_CPF', 'DOCUMENTO', 'COD_DOCUMENTO'], axis=1)

In [151]:
despesa_ceaps.dtypes

ANO                   int64
MES                   int64
SENADOR              object
TIPO_DESPESA         object
FORNECEDOR           object
DATA                 object
DETALHAMENTO         object
VALOR_REEMBOLSADO    object
dtype: object

In [152]:
despesa_ceaps[despesa_ceaps['VALOR_REEMBOLSADO'].str.contains(r'\s', na=False)]

,ANO,MES,SENADOR,TIPO_DESPESA,FORNECEDOR,DATA,DETALHAMENTO,VALOR_REEMBOLSADO
23049,2013,2,RICARDO FERRAÇO,"Passagens aéreas, aquáticas e terrestres nacio...",TAM,04/02/2013,CNPJ: 02.012.862/0001-60 FORNECEDOR: TAM DOCUM...,"1\r\n675,55"


In [153]:
despesa_ceaps.loc[23049, 'VALOR_REEMBOLSADO'] = '1675.55'

In [154]:
despesa_ceaps['VALOR_REEMBOLSADO'] = despesa_ceaps['VALOR_REEMBOLSADO'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float)

In [155]:
despesa_ceaps[['ANO', 'MES']] = despesa_ceaps[['ANO', 'MES']].astype(str)

In [156]:
despesa_ceaps.loc[[57300, 57307, 61736, 63204, 63557, 69939, 69943, 74344,
                    79625, 89872, 91052, 91167, 95412, 110815, 119949, 120180, 
                    122729, 130647, 145166, 147784, 166703, 187713, 193589], ['ANO','DATA']]

,ANO,DATA
57300,2015,"Companhia Aérea: TAM, Localizador: YXGDSJ. Pas..."
57307,2015,"Companhia Aérea: AVIANCA, Localizador: ZNEU9F...."
61736,2015,06/10/2915
63204,2015,26/08/0201
63557,2015,24/02/5015
69939,2015,22/04/0215
69943,2015,23/04/0215
74344,2015,08/05/5201
79625,2016,20/07/5017
89872,2016,02/04/3016


In [157]:
data_errada = [61736, 63204, 63557, 69939, 69943, 74344, 79625, 89872,
                91052, 91167, 95412, 110815, 119949, 120180, 122729, 
                130647, 145166, 147784, 166703, 187713, 193589]

In [158]:
for data in data_errada:
    dia, mes, _ = despesa_ceaps.loc[data, 'DATA'].split('/')
    ano_correto = str(despesa_ceaps.at[data, 'ANO'])
    despesa_ceaps.at[data, 'DATA'] = f'{dia}/{mes}/{ano_correto}'

In [159]:
despesa_ceaps.drop([57300, 57307], inplace=True)

In [160]:
despesa_ceaps['DATA'] = pd.to_datetime(despesa_ceaps['DATA'], format='%d/%m/%Y', errors = 'coerce').dt.date

In [161]:
despesa_ceaps.dtypes

ANO                   object
MES                   object
SENADOR               object
TIPO_DESPESA          object
FORNECEDOR            object
DATA                  object
DETALHAMENTO          object
VALOR_REEMBOLSADO    float64
dtype: object

In [162]:
despesa_ceaps.isnull().sum()

ANO                      0
MES                      0
SENADOR                  0
TIPO_DESPESA             0
FORNECEDOR               0
DATA                     0
DETALHAMENTO         57256
VALOR_REEMBOLSADO        0
dtype: int64

In [163]:
despesa_ceaps.fillna('Nao preenchido', inplace=True)

In [164]:
despesa_ceaps

,ANO,MES,SENADOR,TIPO_DESPESA,FORNECEDOR,DATA,DETALHAMENTO,VALOR_REEMBOLSADO
0,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",CENTRAIS ELÉTRICAS DE RONDÔNIA S.A. - CERON,2013-01-18,Nao preenchido,214.79
1,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",CENTRAIS ELÉTRICAS DE RONDÔNIA S.A. - CERON,2013-01-21,Nao preenchido,57.34
2,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",GILBERTO PISELO DO NASCIMENTO,2013-01-30,Nao preenchido,5000.00
3,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",OI S.A.,2013-01-14,Nao preenchido,398.10
4,2013,1,ACIR GURGACZ,"Locomoção, hospedagem, alimentação, combustíve...",YURI COMÉRCIO DE COMBUSTÍVEIS LTDA,2013-01-23,Nao preenchido,1128.00
...,...,...,...,...,...,...,...,...
225068,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",Exceller Tour,2022-12-06,"Companhia Aérea: LATAM, Localizador: WIXHAI. P...",2893.04
225069,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",Exceller Tour,2022-12-09,"Companhia Aérea: GOL, Localizador: WITOLM. Pas...",1180.19
225070,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",Exceller Tour,2022-12-20,"Companhia Aérea: TAM, Localizador: THPKVQ. Pas...",2671.90
225071,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",Exceller Tour,2022-12-21,"Companhia Aérea: AZUL, Localizador: QNN9HX. Pa...",1334.31


In [165]:
despesa_ceaps.merge
despesa_ceaps.shape

(225071, 8)

In [166]:
despesa_ceaps.to_parquet(data_dir / 'despesa_ceaps.parquet')